# Data mentioned in the text of the Data Overview section
Every number cited in *Data overview* (Figures 5–7 and S3–S6, Tables 16 and S1), all from `cultura_enriched_v3.2`, read from the notebook that draws the figure or table: each cell reruns the preprocessing cells of that notebook (no figure) and prints the numbers the text uses.

In [1]:
import contextlib
import io
import json
import os
import sys
from pathlib import Path

import duckdb
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(50)

HERE = Path.cwd().resolve().parent / 'notebooks_analysis'
NOTEBOOKS = HERE.parent
sys.path.insert(0, str(HERE))


def run_cells(notebook, last_cell, cwd=None):
    """Run the code cells of a notebook up to last_cell and return its variables."""
    path = (HERE / notebook).resolve()
    namespace = {'display': lambda *args, **kwargs: None}
    here = os.getcwd()
    os.chdir(cwd or path.parent)
    sys.path.insert(0, str(path.parent))
    try:
        for i, cell in enumerate(json.loads(path.read_text())['cells'][:last_cell + 1]):
            if cell['cell_type'] == 'code':
                source = ''.join(cell['source'])
                with contextlib.redirect_stdout(io.StringIO()):
                    exec(compile(source, f'{path.name}:{i}', 'exec'), namespace)
    finally:
        os.chdir(here)
        for value in namespace.values():
            if isinstance(value, duckdb.DuckDBPyConnection):
                value.close()
    return namespace


def pct(part, whole):
    return round(100 * part / whole, 1)

## Active individuals per polity at four dates (Figure 5)
From `10_polity_maps`. Per century snapshot: polities drawn, active individuals, and the largest polities with their share of the snapshot.

In [2]:
maps = run_cells('10_polity_maps.ipynb', 9)
drawn = maps['century_counts'].join(maps['polygons'].select('century', 'polity_id').unique(), on=['century', 'polity_id'])
for century in maps['CENTURY_SNAPSHOTS']:
    snapshot = drawn.filter(pl.col('century') == century).sort('n_all', descending=True)
    total = snapshot['n_all'].sum()
    print(f'{century}: {snapshot.height} polities, {total:,} active individuals')
    print(snapshot.head(6).select('polity_name', 'n_all', (pl.col('n_all') / total * 100).round(1).alias('% of snapshot')))

-400: 30 polities, 660 active individuals
shape: (6, 3)
┌────────────────────────┬───────┬───────────────┐
│ polity_name            ┆ n_all ┆ % of snapshot │
│ ---                    ┆ ---   ┆ ---           │
│ str                    ┆ i64   ┆ f64           │
╞════════════════════════╪═══════╪═══════════════╡
│ Macedonian Empire      ┆ 120   ┆ 18.2          │
│ Greek City-States      ┆ 107   ┆ 16.2          │
│ Second Athenian League ┆ 73    ┆ 11.1          │
│ Roman Republic         ┆ 72    ┆ 10.9          │
│ Achaemenid Empire      ┆ 68    ┆ 10.3          │
│ Peloponnesian League   ┆ 56    ┆ 8.5           │
└────────────────────────┴───────┴───────────────┘
500: 67 polities, 2,132 active individuals
shape: (6, 3)
┌───────────────────────┬───────┬───────────────┐
│ polity_name           ┆ n_all ┆ % of snapshot │
│ ---                   ┆ ---   ┆ ---           │
│ str                   ┆ i64   ┆ f64           │
╞═══════════════════════╪═══════╪═══════════════╡
│ Eastern Roman Empire  ┆

## Occupations over time (Figure S3)
From `03_occupations_over_time`. Mean share of active individuals per occupation over the 50-year buckets of each period (shares include the *Other* category).

In [3]:
occupations = run_cells('03_occupations_over_time.ipynb', 15)


def mean_shares(scope, low, high, top=6):
    shares = occupations[f'{scope}_shares'].filter(pl.col('bucket').is_between(low, high))
    order = occupations[f'{scope}_order']
    means = shares.select(order).mean().transpose(include_header=True, header_name='occupation', column_names=['share'])
    return means.with_columns((pl.col('share') * 100).round(1)).sort('share', descending=True).head(top)


for scope, low, high in [('world', -500, 1699), ('world', 1700, 1999), ('world', 1900, 1999), ('roman', -3000, 3000), ('tang', -3000, 3000)]:
    print(scope, low, high)
    print(mean_shares(scope, low, high))
print(occupations['world_shares'].filter(pl.col('bucket') >= 1800).select('bucket', *[pl.col(o).mul(100).round(1) for o in ['politician', 'writer', 'university teacher', 'actor', 'researcher']]))

world -500 1699
shape: (6, 2)
┌────────────────────┬───────┐
│ occupation         ┆ share │
│ ---                ┆ ---   │
│ str                ┆ f64   │
╞════════════════════╪═══════╡
│ Other              ┆ 50.0  │
│ politician         ┆ 17.3  │
│ military personnel ┆ 8.6   │
│ Catholic priest    ┆ 5.9   │
│ poet               ┆ 5.1   │
│ writer             ┆ 4.9   │
└────────────────────┴───────┘
world 1700 1999
shape: (6, 2)
┌────────────────────┬───────┐
│ occupation         ┆ share │
│ ---                ┆ ---   │
│ str                ┆ f64   │
╞════════════════════╪═══════╡
│ Other              ┆ 62.3  │
│ politician         ┆ 7.7   │
│ writer             ┆ 5.3   │
│ painter            ┆ 5.1   │
│ university teacher ┆ 2.1   │
│ poet               ┆ 1.9   │
└────────────────────┴───────┘
world 1900 1999
shape: (6, 2)
┌────────────────────┬───────┐
│ occupation         ┆ share │
│ ---                ┆ ---   │
│ str                ┆ f64   │
╞════════════════════╪═══════╡
│ Other    

## Comparison with existing databases (Table 16)
From `07_comparison_with_other_databases`.

In [4]:
comparison = run_cells('07_comparison_with_other_databases.ipynb', 20)
cultura, cv = comparison['cultura_counts'], comparison['cv_counts']
stats = comparison['cultura_stats'].row(0, named=True)
total = cultura['total']
print(f"Cultura {total:,}; x HBR {total / 7_015_353:.2f}; x CV {total / cv['total']:.2f}; x Pantheon 2.0 {total / comparison['pantheon_2_counts']['total']:.1f}")
print(f"CV found in Cultura: {pct(stats['in_cross_verified'], cv['total'])}%; Pantheon 2.0 found: {pct(stats['in_pantheon_2'], comparison['pantheon_2_counts']['total'])}%")
for key in ['with_polity', 'with_floruit', 'with_occupation', 'outside_cv_seven', 'nw_wikipedia_only', 'nw_catalog_only']:
    print(f'{key}: {cultura[key]:,} ({pct(cultura[key], total)}%)')
print(f"HBR: with dates {pct(4_239_378, 7_015_353)}%, with occupation {pct(5_324_653, 7_015_353)}%")
comparison['comparison_table']

Cultura 13,002,840; x HBR 1.85; x CV 5.67; x Pantheon 2.0 102.7
CV found in Cultura: 99.1%; Pantheon 2.0 found: 99.0%
with_polity: 3,631,573 (27.9%)
with_floruit: 5,072,222 (39.0%)
with_occupation: 9,038,068 (69.5%)
outside_cv_seven: 1,582,613 (12.2%)
nw_wikipedia_only: 938,927 (7.2%)
nw_catalog_only: 261,325 (2.0%)
HBR: with dates 60.4%, with occupation 75.9%


dimension,cultura,cross_verified,pantheon_2,hbr,schich_2014,chaney_2024
str,str,str,str,str,str,str
"""Total individuals""","""13,002,840""","""2,291,817""","""126,582""","""7,015,353""","""120,211""","""537,247"""
"""With life-stage polity / citizenship""","""3,631,573""","""2,238,318""","""123,104""","""5,443,212""","""—""","""486,179"""
"""With active life / birth-or-death date""","""5,072,222""","""2,140,640""","""125,324""","""4,239,378""","""120,211""","""537,247"""
"""With occupation""","""9,038,068""","""2,276,400""","""126,520""","""5,324,653""","""—""","""537,247"""
"""In any of the seven main CV Wikipedia editions""","""3,161,305""","""2,290,997""","""—""","""—""","""—""","""—"""
"""In Wikipedia outside the seven main CV editions""","""1,582,613""","""820""","""—""","""—""","""—""","""—"""
"""Only in non-Western Wikipedia""","""938,927""","""169""","""—""","""—""","""—""","""—"""
"""Only in non-Western catalogs""","""261,325""","""—""","""—""","""—""","""—""","""—"""


## Chinese polities in Cultura and Cross-Verified (Table S1)
From `05_chinese_polities_vs_cross_verified`. Subtotal of each panel, as printed under the tables.

In [5]:
chinese = run_cells('05_chinese_polities_vs_cross_verified.ipynb', 13)


def panel_total(rows, field):
    return sum(int(row[field].replace(',', '')) for row in rows if row[field])


panels = {label: chinese[f'panel_{label}'] for label in 'abcd'}
for label, rows in panels.items():
    print(f"({label}) Cultura {panel_total(rows, 'cu_n'):,}  CV {panel_total(rows, 'cv_n'):,}")
print(f"all panels: Cultura {sum(panel_total(r, 'cu_n') for r in panels.values()):,}  CV {sum(panel_total(r, 'cv_n') for r in panels.values()):,}")
rows = pl.DataFrame(chinese['flat_rows']).drop('sort_key')
rows.filter(pl.col('cu_polity').is_in(['Tang Dynasty', 'Ming Dynasty', 'Qing Dynasty']))

(a) Cultura 1,827  CV 331
(b) Cultura 3,091  CV 146
(c) Cultura 38,438  CV 625
(d) Cultura 102,365  CV 17,455
all panels: Cultura 145,721  CV 18,557


cu_polity,cu_period,cu_n,cv_token,cv_period,cv_n
str,str,str,str,str,str
"""Tang Dynasty""","""623–910""","""25,647""","""Tang dynasty""","""618–907""","""268"""
"""Ming Dynasty""","""1375–1644""","""16,571""","""Ming dynasty""","""1368–1644""","""288"""
"""Qing Dynasty""","""1645–1911""","""32,215""","""Qing dynasty""","""1636–1912""","""768"""


## Cultura and Cross-Verified over time and by region (Figure 6)
From `06_coverage_vs_cross_verified`. Ratio of Cultura to Cross-Verified individuals per period, globally and per region.

In [6]:
coverage = run_cells('06_coverage_vs_cross_verified.ipynb', 19)


def ratio(cultura, cross_verified):
    return cultura.join(cross_verified, on='continent').with_columns((pl.col('cultura') / pl.col('cross_verified')).round(1).alias('ratio')).sort('ratio', descending=True)


per_bin = coverage['per_bin_global']
for low, high in [(-1500, 1499), (1500, 1899), (1900, 2099)]:
    period = per_bin.filter(pl.col('bin').is_between(low, high)).select(pl.sum('cultura', 'cross_verified'))
    print(f'{low}–{high}: Cultura {period["cultura"][0]:,}, CV {period["cross_verified"][0]:,}, ratio {period["cultura"][0] / period["cross_verified"][0]:.1f}')
for low, high in [(-1500, 1499), (1500, 1899)]:
    by_region = [coverage[name].filter(pl.col('bin').is_between(low, high)).group_by('continent').agg(pl.sum('n').alias(column))
                 for name, column in [('cultura_bins_continent', 'cultura'), ('cv_bins_continent', 'cross_verified')]]
    print(f'{low}–{high}')
    print(ratio(*by_region))

-1500–1499: Cultura 203,888, CV 54,174, ratio 3.8
1500–1899: Cultura 1,429,322, CV 364,165, ratio 3.9
1900–2099: Cultura 4,289,560, CV 1,216,389, ratio 3.5
-1500–1499
shape: (6, 4)
┌───────────────┬─────────┬────────────────┬───────┐
│ continent     ┆ cultura ┆ cross_verified ┆ ratio │
│ ---           ┆ ---     ┆ ---            ┆ ---   │
│ str           ┆ i64     ┆ i64            ┆ f64   │
╞═══════════════╪═════════╪════════════════╪═══════╡
│ Asia          ┆ 55566   ┆ 6290           ┆ 8.8   │
│ Africa        ┆ 1212    ┆ 474            ┆ 2.6   │
│ Middle East   ┆ 10842   ┆ 4590           ┆ 2.4   │
│ Europe        ┆ 59264   ┆ 33237          ┆ 1.8   │
│ North America ┆ 85      ┆ 53             ┆ 1.6   │
│ Latin America ┆ 350     ┆ 215            ┆ 1.6   │
└───────────────┴─────────┴────────────────┴───────┘
1500–1899
shape: (6, 4)
┌───────────────┬─────────┬────────────────┬───────┐
│ continent     ┆ cultura ┆ cross_verified ┆ ratio │
│ ---           ┆ ---     ┆ ---            ┆ ---   │


## Temporal precision of Chinese dynasties (Figure S4)
From `04_chinese_dynasties_precision`.

In [7]:
dynasties = run_cells('04_chinese_dynasties_precision.ipynb', 12)
precision = dynasties['precision_counts']
print(precision.with_columns([(pl.col(c) / pl.col('total') * 100).round(1).alias(f'{c} %') for c in ['year', 'century', 'none']]))
totals = precision.select(pl.sum('year', 'decade', 'century', 'none', 'total'))
print(f'year precision over all dynasties: {pct(totals["year"][0], totals["total"][0])}% of {totals["total"][0]:,}')

shape: (24, 9)
┌────────────────────────┬───────┬────────┬─────────┬───┬───────┬────────┬───────────┬────────┐
│ dynasty                ┆ year  ┆ decade ┆ century ┆ … ┆ total ┆ year % ┆ century % ┆ none % │
│ ---                    ┆ ---   ┆ ---    ┆ ---     ┆   ┆ ---   ┆ ---    ┆ ---       ┆ ---    │
│ str                    ┆ i64   ┆ i64    ┆ i64     ┆   ┆ i64   ┆ f64    ┆ f64       ┆ f64    │
╞════════════════════════╪═══════╪════════╪═════════╪═══╪═══════╪════════╪═══════════╪════════╡
│ Shang Dynasty          ┆ 5     ┆ 0      ┆ 3       ┆ … ┆ 8     ┆ 62.5   ┆ 37.5      ┆ 0.0    │
│ Zhou Dynasty           ┆ 16    ┆ 0      ┆ 5       ┆ … ┆ 21    ┆ 76.2   ┆ 23.8      ┆ 0.0    │
│ Qin Dynasty            ┆ 46    ┆ 2      ┆ 30      ┆ … ┆ 78    ┆ 59.0   ┆ 38.5      ┆ 0.0    │
│ Han Dynasty            ┆ 479   ┆ 20     ┆ 491     ┆ … ┆ 990   ┆ 48.4   ┆ 49.6      ┆ 0.0    │
│ Xin Dynasty            ┆ 40    ┆ 0      ┆ 47      ┆ … ┆ 87    ┆ 46.0   ┆ 54.0      ┆ 0.0    │
│ Western Jin            

## Individuals dated only to the century (paragraph *Dates known only to the century*, Table 9)
From `scripts/4-database_enrichment/01_notebooks_dates/05_coarse_dates`, cells up to the count (the notebook saves a parquet later, which is not run here). Shares are of all individuals.

In [8]:
coarse_dates = run_cells('../../scripts/4-database_enrichment/01_notebooks_dates/05_coarse_dates.ipynb', 3)
coarse = coarse_dates['coarse']
print(f'{coarse.height:,} individuals have only century- or millennium-level dates: {pct(coarse.height, cultura["total"])}% of all {cultura["total"]:,} individuals')

108,018 individuals have only century- or millennium-level dates: 0.8% of all 13,002,840 individuals


## Individuals without a polity (Figure S5)
The headline share is of all individuals: individuals with an active life but no life-stage polity, read from `cultura_enriched_v3.2`. The century shares come from `11_polity_match_gaps` and are within each century, among individuals with an active life.

In [9]:
with duckdb.connect('../../data/cultura/cultura_enriched_v3.2.duckdb', read_only=True) as con:
    no_polity = con.sql('''
        SELECT count(*) FROM individual_enriched
        WHERE start_of_active_life.year IS NOT NULL
          AND coalesce(len(pre_active_life_polity), 0) + coalesce(len(active_life_polity), 0) + coalesce(len(post_active_life_polity), 0) = 0
    ''').fetchone()[0]
print(f'active life but no life-stage polity: {no_polity:,}, {pct(no_polity, cultura["total"])}% of all individuals')

gaps = run_cells('11_polity_match_gaps.ipynb', 12)
share = gaps['share_with_polity'].with_columns((100 - pl.col('pct_with_polity')).round(1).alias('pct_without'))
for low, high in [(-3000, -501), (-300, 299), (600, 799), (900, 1799), (1800, 1899), (1900, 1999)]:
    period = share.filter(pl.col('bucket').is_between(low, high))
    print(f'{low}–{high}: without a polity, among individuals with an active life {period["pct_without"].min()}–{period["pct_without"].max()}%')
share.select('bucket', 'individuals', 'pct_without')

active life but no life-stage polity: 1,514,341, 11.6% of all individuals


-3000–-501: without a polity, among individuals with an active life 37.0–52.5%
-300–299: without a polity, among individuals with an active life 23.3–27.5%
600–799: without a polity, among individuals with an active life 14.6–23.9%
900–1799: without a polity, among individuals with an active life 44.2–57.6%
1800–1899: without a polity, among individuals with an active life 35.5–35.5%
1900–1999: without a polity, among individuals with an active life 25.7–25.7%


bucket,individuals,pct_without
i32,i64,f64
-1400,59,52.5
-900,83,48.2
-800,122,37.7
-700,212,41.5
-600,384,37.0
-500,712,54.5
-400,833,38.5
-300,823,23.3
-200,968,27.5


## Catalogs and Western bias (Figures 7 and S6)
From `02_catalog_coverage`. Figure 7: individuals born 1300–1849 covered only by Western or only by non-Western catalogs or Wikipedias, per polity world (Chinese world, Japan, Korea).

In [10]:
catalogs = run_cells('02_catalog_coverage.ipynb', 22)
print(f'identifier properties: {catalogs["identifier_properties"].height:,}')
print(catalogs['catalog_categories'].group_by('category').len().sort('len', descending=True))
flows = catalogs['sankey_flows']
print(flows.group_by('catalog_group').agg(pl.sum('individual_count')).with_columns((pl.col('individual_count') / flows['individual_count'].sum() * 100).round(1).alias('%')).sort('individual_count', descending=True))
print(catalogs['dynasty_ranking'].head(5))
panel = catalogs['world_panel'].group_by('world_name').agg(pl.sum('Western only'), pl.sum('Non-Western only')).sort('world_name')
print(panel.with_columns((pl.col('Non-Western only') / pl.col('Western only')).round(1).alias('non-Western / Western')))
per_bucket = catalogs['world_panel'].with_columns((pl.col('Non-Western only') / pl.col('Western only')).round(1).alias('ratio'))
print('smallest ratio per world over the 50-year periods:')
print(per_bucket.group_by('world_name').agg(pl.min('ratio')).sort('world_name'))

identifier properties: 10,198
shape: (4, 2)
┌───────────────┬──────┐
│ category      ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ western       ┆ 4417 │
│ null          ┆ 4235 │
│ non_western   ┆ 1545 │
│ international ┆ 1    │
└───────────────┴──────┘
shape: (13, 3)
┌──────────────────────────────────────────┬──────────────────┬──────┐
│ catalog_group                            ┆ individual_count ┆ %    │
│ ---                                      ┆ ---              ┆ ---  │
│ str                                      ┆ u32              ┆ f64  │
╞══════════════════════════════════════════╪══════════════════╪══════╡
│ Google Knowledge Graph ID                ┆ 16864            ┆ 28.2 │
│ Geni.com profile ID                      ┆ 14368            ┆ 24.0 │
│ Other catalogs                           ┆ 4494             ┆ 7.5  │
│ VIAF cluster ID                          ┆ 3661             ┆ 6.1  │
│ Yale LUX ID                              ┆ 3361   